# This notebook is to host embedding **Ollama** models on **kaggle** and be able to use them with **ngrok**



### Make sure to use the kaggle GPUs although the embedding model might not need it

### Create an account on ngrok visit https://dashboard.ngrok.com/get-started/setup copy Your Authtoken, then here on kaggle 
### *Add-ons -> Secrets -> Add Secres* 
### label it **NGROK_AUTH_TOKEN**, paste your token in the Value field, and save it.

In [1]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 83 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (379 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
Selecting previously unselected package zstd.
(Reading database ... 121026 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.

In [2]:
!sudo apt update && sudo apt install pciutils lshw

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [114 kB]m
Get:4 https://cli.github.com/packages stable/main amd64 Packages [359 B]       
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]3m
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [11.0 MB]  
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,318 kB]
Get:13 http

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                        0.6%                                                       4.3%                        13.3%  23.6%       69.4%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!pip install -q pyngrok ollama

In [6]:
import subprocess
import time
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# 1. Start the Ollama engine in the background
subprocess.Popen(['ollama', 'serve'])
print("Starting Ollama server...")
time.sleep(3) # Give the server a few seconds to boot

# 2. Authenticate ngrok using your Kaggle Secret
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

# 3. Open the tunnel to port 11434
tunnel = ngrok.connect("11434", host_header="localhost:11434")
print(f"Success! Your public API URL is: {tunnel.public_url}")

Starting Ollama server...


Error: listen tcp 127.0.0.1:11434: bind: address already in use


Success! Your public API URL is: https://dividing-supernova-hardener.ngrok-free.dev                 


### copy your API URL to use it on your device

In [7]:
import ollama

# Connect the client to your new public URL
client = ollama.Client(host=tunnel.public_url)

# Define the embedding model
embedding_model = 'qwen3-embedding:6b'

# Download the model to the Kaggle GPU
print(f"Downloading {embedding_model}...")
client.pull(embedding_model)

# Send a text string to generate its vector representation
print("Generating embedding...")
response = client.embeddings(
    model=embedding_model,
    prompt='Write a haiku about Python.'
)

# The response contains the vector array, not text
vector = response['embedding']

print(f"Success! Generated a vector with {len(vector)} dimensions.")
# Print the first 5 numbers just to see what it looks like
print(f"First 5 values: {vector[:5]}")

[GIN] 2026/09/24 - 02:29:24 | 500 |  702.693715ms |    35.221.162.4 | POST     "/api/pull"


ResponseError: pull model manifest: file does not exist (status code: 500)

### Simple example on how to use your API

In [ ]:
import ollama
import json

# 1. Connect to your active Kaggle URL
KAGGLE_URL = "https://museum-action-backtrack.ngrok-free.dev"
client = ollama.Client(host=KAGGLE_URL)
embedding_model = 'qwen3-embedding:6b'

# 2. Your document chunks (Replace this with your actual chunks list)
chunks = [
    "Machine learning is a subset of artificial intelligence.",
    "Python is a popular programming language for data science.",
    "Embeddings convert human text into mathematical vectors."
]

# 3. Create a list to store our processed data
embedded_database = []

print(f"Sending {len(chunks)} chunks to Kaggle...")

# 4. Feed the chunks to the API
for index, text in enumerate(chunks):
    # Send the request over the tunnel
    response = client.embeddings(
        model=embedding_model,
        prompt=text
    )
    
    # Bundle the original text and its new vector together
    embedded_database.append({
        "id": index,
        "text": text,
        "vector": response['embedding']
    })
    
    print(f"Processed chunk {index + 1}/{len(chunks)}")

print("\nSuccess! All chunks embedded.")

# 5. (Optional but recommended) Save the results to a file so you don't lose them
with open('embedded_chunks.json', 'w') as file:
    json.dump(embedded_database, file)

print("Saved to embedded_chunks.json")